# 合并与分割

合并是指将多个张量在某个维度上合并为一个张量。以某学校班级成绩册数据为例，
设张量𝑨保存了某学校 1~4 号班级的成绩册，每个班级 35 个学生，共 8 门科目成绩，则张
量𝑨的 shape 为：[4,35,8]；同样的方式，张量𝑩保存了其它 6 个班级的成绩册，shape 为
[6,35,8]。通过合并这 2 份成绩册，便可得到学校所有班级的成绩册，记为张量𝑪，shape 应
为[10,35,8]，其中，10 代表 10 个班级，35 代表 35 个学生，8 代表 8 门科目。这就是张量
合并的意义所在。

## 1.拼接 tf.concat(tensors, axis)  
在 TensorFlow 中，可以通过 tf.concat(tensors, axis)函数拼接张量，其中参数
tensors 保存了所有需要合并的张量 List，axis 参数指定需要合并的维度索引。回到上面的
例子，我们在班级维度上合并成绩册，这里班级维度索引号为 0，即 axis=0，

In [2]:
import tensorflow as tf
a = tf.random.uniform([4, 35, 8], maxval=100, dtype=tf.int32)  # 模拟成绩册 A
b = tf.random.uniform([6, 35, 8], maxval=100, dtype=tf.int32)  # 模拟成绩册 B
tf.concat([a, b], axis=0).shape

TensorShape([10, 35, 8])

除了可以在班级维度上进行拼接合并，还可以在其他维度上拼接合并张量。考虑张量𝑨保存了所有班级的所有学生的前 4 门科目成绩，shape 为[10,35,4]，张量𝑩保存了剩下的 4
门科目成绩，shape 为[10,35,4]，则可以拼接合并 shape 为[10,35,8]的总成绩册张量，实现
如下：

In [3]:
a = tf.random.normal([10, 35, 4])
b = tf.random.normal([10, 35, 4])
tf.concat([a, b], axis=2).shape

TensorShape([10, 35, 8])

从语法上来说，拼接合并操作可以在任意的维度上进行，**唯一的约束是非合并维度的长度必须一致**。
比如 shape 为[4,32,8]和 shape 为[6,35,8]的张量不能直接在班级维度上进行
合并，因为学生数量维度的长度并不一致，一个为 32，另一个为 35.

## 2.堆叠 tf.stack(tensors, axis)  
拼接操作直接在现有维度上合并数据，并不会创建新的维度。如果在合并数据
时，希望创建一个新的维度，则需要使用 tf.stack 操作。

考虑张量𝑨保存了某个班级的成绩
册，shape 为[35,8]，张量𝑩保存了另一个班级的成绩册，shape 为[35,8]。合并这 2 个班级
的数据时，则需要创建一个新维度，定义为班级维度，新维度可以选择放置在任意位置，
一般根据大小维度的经验法则，将较大概念的班级维度放置在学生维度之前，则合并后的
张量的新 shape 应为[2,35,8]。

使用 tf.stack(tensors, axis)可以堆叠方式合并多个张量，通过 tensors 列表表示，参数
axis 指定新维度插入的位置，axis 的用法与 tf.expand_dims 的一致，当axis ≥ 0时，在 axis
之前插入；当axis < 0时，在 axis 之后插入新维度。例如 shape 为[𝑏, 𝑐, ℎ, 𝑤]的张量，在不
同位置通过 stack 操作插入新维度

In [4]:
# 堆叠方式合并这 2 个班级成绩册，班级维度插入在 axis=0 位置
a = tf.random.normal([35, 8])
b = tf.random.normal([35, 8])
tf.stack([a, b], axis=0).shape

TensorShape([2, 35, 8])

In [5]:
# 同样可以选择在其他位置插入新维度，例如，最末尾插入班级维度：
a = tf.random.normal([35, 8])
b = tf.random.normal([35, 8])
tf.stack([a, b], axis=-1).shape

TensorShape([35, 8, 2])

tf.stack 也需要满足张量堆叠合并条件，它需要**所有待合并的张量 shape 完全一致才可合并**。

## 3.分割 tf.split(x, num_or_size_splits, axis)  
合并操作的逆过程就是分割，将一个张量分拆为多个张量。继续考虑成绩册的例子，
我们得到整个学校的成绩册张量，shape 为[10,35,8]，现在需要将数据在班级维度切割为
10 个张量，每个张量保存了对应班级的成绩册数据。

**num_or_size_splits 参数：切割方案。当 num_or_size_splits 为单个数值时，如 10，表
示等长切割为 10 份；当 num_or_size_splits 为 List 时，List 的每个元素表示每份的长
度，如[2,4,2,2]表示切割为 4 份，每份的长度依次是 2、4、2、2。**  
axis 参数：指定分割的维度索引号

In [10]:
# 我们将总成绩册张量切割为 10 份
x = tf.random.normal([10, 35, 8])
result = tf.split(x, num_or_size_splits=10, axis=0)  # 为了独立处理所有⼦张量所以返回的列表
len(result)

10

In [11]:
result[0].shape  # 查看第一个班级的成绩册张量

TensorShape([1, 35, 8])

进行不等长的切割，例如，将数据切割为 4 份，每份长度分别为[4,2,2,2]

In [13]:
x = tf.random.normal([10, 35, 8])
result = tf.split(x, [4, 2, 2, 2], axis=0)
print('result_len:', len(result))
result[0].shape

result_len: 4


TensorShape([4, 35, 8])

### 长度为 1 的方式分割 tf.unstack(x,axis)  
特别地，如果希望在某个维度上全部按长度为 1 的方式分割，**还可以使用 tf.unstack(x,axis)函数**。这种方式是 tf.split 的一种特殊情况，切割长度固定为 1，只需要指定切割维度
的索引号即可。例如，将总成绩册张量在班级维度进行 unstack 操作

In [15]:
x = tf.random.normal([10, 35, 8])
result = tf.unstack(x, axis=0)
print('result_len:',len(result))
result[0].shape

result_len: 10


TensorShape([35, 8])